In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
import torchvision
import os
import numpy as np
from PIL import Image

In [ ]:
class PairedCIFAR10(Dataset):
    def __init__(self, clean_path, corrupted_path, transform=None, corruption_type="gaussian_noise", severity=1):
        self.transform = transform

        clean_data = torchvision.datasets.CIFAR10(
            root=clean_path,
            train=False,
            download=True
        )

        corrupted_data_path = os.path.join(corrupted_path, corruption_type + ".npy")
        full_corrupted_data = torch.from_numpy(np.load(corrupted_data_path))

        num_per_severity = 10000
        start_idx = (severity - 1) * num_per_severity
        end_idx = severity * num_per_severity
        self.corrupted_data = full_corrupted_data[start_idx:end_idx]
        self.clean_data = [clean_data[i][0] for i in range(len(clean_data))]
        self.labels = [clean_data[i][1] for i in range(len(clean_data))]

    def __len__(self):
        return len(self.clean_data)
    
    def __getitem__(self, idx):
        clean_img = self.clean_data[idx]
        corrupted_img = self.corrupted_data[idx]
        label = self.labels[idx]

        # 将 corrupted_img 从 Tensor 转换为 PIL Image
        # corrupted_img 的形状是 [H, W, C]，需要转换为 uint8
        corrupted_img_np = corrupted_img.numpy().astype(np.uint8)
        corrupted_img = Image.fromarray(corrupted_img_np)

        if self.transform:
            clean_img = self.transform(clean_img)
            corrupted_img = self.transform(corrupted_img)

        return corrupted_img, clean_img, label

In [ ]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

clean_path = 'Task2/data'
corrupted_path = "MTL/data/CIFAR-10-C"

# 随机抽取样本,并以图片形式展示检测是否一致
dataset = PairedCIFAR10(clean_path, corrupted_path, transform=transform, corruption_type="gaussian_noise", severity=3)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
for corrupted_img, clean_img, label in dataloader:
    print(f"Label: {label.item()}")
    corrupted_img = corrupted_img.squeeze(0).permute(1, 2, 0).numpy()
    clean_img = clean_img.squeeze(0).permute(1, 2, 0).numpy()

    corrupted_img = (corrupted_img * np.array([0.2470, 0.2435, 0.2616])) + np.array([0.4914, 0.4822, 0.4465])
    clean_img = (clean_img * np.array([0.2470, 0.2435, 0.2616])) + np.array([0.4914, 0.4822, 0.4465])

    corrupted_img = np.clip(corrupted_img * 255, 0, 255).astype(np.uint8)
    clean_img = np.clip(clean_img * 255, 0, 255).astype(np.uint8)

    corrupted_pil = Image.fromarray(corrupted_img)
    clean_pil = Image.fromarray(clean_img)

    corrupted_pil.show(title="Corrupted Image")
    clean_pil.show(title="Clean Image")
    break